In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SINGLE GPU OPTIMIZED MULTIMODAL DISASTER CLASSIFIER  v3
# Swin-T + BanglaBERT — BIDIRECTIONAL CROSS-ATTENTION FUSION
#
# NEW IN v3 (over v2):
#   1. CutMix  — batch-level augmentation via collate_fn (prob=0.5).
#                λ sampled from Beta(1,1); loss mixed proportionally.
#                img_valid AND-ed across paired samples.
#   2. Full PNG/RGBA support — RGBA images alpha-composited onto white
#                background; image_id_to_path probes .png explicitly.
#
# v2 features retained:
#   1. EMA  — Exponential Moving Average (decay=0.999).
#   2. LLRD — Layer-wise Learning Rate Decay (alpha=0.85).
#   3. TTA  — 5-view Test-Time Augmentation.
#   4. 5-Fold Stratified CV.
#
# Optimized for: Kaggle T4 x1 (16 GB VRAM), AMP fp16
# ════════════════════════════════════════════════════════════════════════

import os
import gc
import re
import copy
import string
import shutil
import warnings

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast

import torchvision.transforms as T
from torchvision.io import read_image
from torchvision.models import swin_t, Swin_T_Weights

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from tqdm.auto import tqdm

# ════════════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════════════

class CFG:
    SOURCE_DIR   = "/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0/Dataset/Image/Train"
    TARGET_DIR   = "/kaggle/working/disaster_dataset"
    TRAIN_CSV    = "/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0/Dataset/CSV Dataset/Disaster_train.csv"
    TEST_CSV     = "/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0/Dataset/CSV Dataset/Disaster_test.csv"
    TEST_IMG_DIR = "/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0/Dataset/Image/Test"
    OUTPUT_DIR   = "/kaggle/working"

    TEXT_MODEL   = "csebuetnlp/banglabert"

    IMG_SIZE     = 224
    MAX_LEN      = 96
    NUM_CLASSES  = 8

    # ── Cross-attention fusion dims ──────────────────────────────────
    FUSION_DIM   = 512
    ATTN_HEADS   = 8

    BATCH_SIZE   = 24
    EPOCHS       = 10

    # Base LRs — LLRD will scale these per-layer
    TXT_LR       = 2e-5
    IMG_LR       = 3e-5
    HEAD_LR      = 1e-4

    # LLRD alpha — each layer gets alpha * lr of the layer above it
    LLRD_ALPHA   = 0.85   # 0.85^12 ≈ 0.14× at the embedding layer

    # EMA decay
    EMA_DECAY    = 0.999

    # 5-Fold CV
    N_FOLDS      = 5

    # CutMix probability per batch
    CUTMIX_PROB  = 0.5

    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    GRAD_CLIP    = 1.0
    NUM_WORKERS  = 2
    AMP          = True
    SEED         = 42
    PATIENCE     = 3
    LABEL_SMOOTH = 0.05
    DROPOUT      = 0.3
    DEVICE       = "cuda"

# ════════════════════════════════════════════════════════════════════════
# SEED
# ════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = True

set_seed(CFG.SEED)

# ════════════════════════════════════════════════════════════════════════
# TEXT CLEANING & IMAGE ORGANISATION
# ════════════════════════════════════════════════════════════════════════

def clean_bangla(text):
    if pd.isnull(text): return ""
    text = str(text)
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[0-9০-৯]+', ' ', text)
    punct = string.punctuation + "\u2018\u2019\u201c\u201d\u2026\u2014\u2013\u0964"
    text = text.translate(str.maketrans('', '', punct))
    text = re.sub(r'[^a-zA-Z\u0980-\u09FF\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

CLASS_MAP = {
    "landslides":    "Landslides",
    "tropical_storm":"Tropical Storm",
    "drought":       "Drought",
    "flood":         "Flood",
    "non_disaster":  "Non Disaster",
    "earthquake":    "Earthquake",
    "human_damage":  "Human Damage",
    "wildfire":      "Wildfire",
}

def organise_images():
    for cls in CLASS_MAP.values():
        os.makedirs(os.path.join(CFG.TARGET_DIR, cls), exist_ok=True)
    for fname in os.listdir(CFG.SOURCE_DIR):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")): continue
        for key, cls in CLASS_MAP.items():
            if fname.startswith(key):
                src = os.path.join(CFG.SOURCE_DIR, fname)
                dst = os.path.join(CFG.TARGET_DIR, cls, fname)
                if not os.path.exists(dst): shutil.copy(src, dst)
                break

organise_images()

def build_path_map():
    path_map = {}
    for directory in [CFG.TARGET_DIR, CFG.SOURCE_DIR, CFG.TEST_IMG_DIR]:
        for root, _, files in os.walk(directory):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    stem = os.path.splitext(f)[0]
                    if stem not in path_map:
                        path_map[stem] = os.path.join(root, f)
    return path_map

PATH_MAP = build_path_map()

def image_id_to_path(image_id):
    """
    Resolve an image_id string to a file path.

    Priority:
      1. Strip any known extension and look up the stem in PATH_MAP
         (covers .jpg / .jpeg / .png already indexed at startup).
      2. If not found, probe stem + each extension explicitly —
         handles CSVs where image_id carries no extension, or where
         a PNG was stored without one.
    """
    stem = str(image_id)
    for ext in (".jpg", ".jpeg", ".png"):
        if stem.lower().endswith(ext):
            stem = stem[: -len(ext)]
            break

    if stem in PATH_MAP:
        return PATH_MAP[stem]

    # Explicit extension probes (PNG first, then JPEG variants)
    for ext in (".png", ".jpg", ".jpeg"):
        probe = stem + ext
        if probe in PATH_MAP:
            return PATH_MAP[probe]

    return None   # triggers zero-tensor fallback in load_image

# ════════════════════════════════════════════════════════════════════════
# LOAD CSV & TOKENIZE
# ════════════════════════════════════════════════════════════════════════

train_df = pd.read_csv(CFG.TRAIN_CSV)
test_df  = pd.read_csv(CFG.TEST_CSV)

train_df["clean_text"] = train_df["context"].apply(clean_bangla)
test_df["clean_text"]  = test_df["context"].apply(clean_bangla)

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["category"])

tokenizer    = AutoTokenizer.from_pretrained(CFG.TEXT_MODEL)
train_tokens = tokenizer(
    train_df["clean_text"].tolist(),
    max_length=CFG.MAX_LEN, padding="max_length", truncation=True,
)
test_tokens  = tokenizer(
    test_df["clean_text"].tolist(),
    max_length=CFG.MAX_LEN, padding="max_length", truncation=True,
)

TRAIN_IDS  = np.array(train_tokens["input_ids"],      dtype=np.int32)
TRAIN_MASK = np.array(train_tokens["attention_mask"],  dtype=np.int32)
TEST_IDS   = np.array(test_tokens["input_ids"],        dtype=np.int32)
TEST_MASK  = np.array(test_tokens["attention_mask"],   dtype=np.int32)

# ════════════════════════════════════════════════════════════════════════
# AUGMENTATIONS
# ════════════════════════════════════════════════════════════════════════

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.Normalize(mean=_MEAN, std=_STD),
])

val_transforms = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.Normalize(mean=_MEAN, std=_STD),
])

# ════════════════════════════════════════════════════════════════════════
# TTA TRANSFORMS
#
#  5 views per image, averaged at inference:
#    0 — original (clean resize)
#    1 — horizontal flip
#    2 — random crop (scale 0.85–1.0, ratio 0.9–1.1)
#    3 — random crop (scale 0.80–0.95, ratio 0.9–1.1)
#    4 — random crop (scale 0.80–0.95, ratio 0.9–1.1)  different seed
# ════════════════════════════════════════════════════════════════════════

tta_transforms = [
    # View 0 — clean baseline (same as val)
    T.Compose([
        T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        T.Normalize(mean=_MEAN, std=_STD),
    ]),
    # View 1 — horizontal flip
    T.Compose([
        T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        T.RandomHorizontalFlip(p=1.0),
        T.Normalize(mean=_MEAN, std=_STD),
    ]),
    # Views 2–4 — random resized crops at different scales
    T.Compose([
        T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
        T.Normalize(mean=_MEAN, std=_STD),
    ]),
    T.Compose([
        T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.80, 0.95), ratio=(0.9, 1.1)),
        T.Normalize(mean=_MEAN, std=_STD),
    ]),
    T.Compose([
        T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.80, 0.95), ratio=(0.9, 1.1)),
        T.Normalize(mean=_MEAN, std=_STD),
    ]),
]

# ════════════════════════════════════════════════════════════════════════
# CUTMIX AUGMENTATION
#
#  CutMix randomly pastes a rectangular patch from one image onto another
#  and mixes their labels proportionally to the actual patch area.
#
#  Implementation notes:
#    - Applied as a collate_fn wrapper so it operates on whole batches
#      (required because we need to mix pairs of samples).
#    - CUTMIX_PROB controls per-batch application rate.
#    - Loss: lam * CE(pred, y_a) + (1 - lam) * CE(pred, y_b)
#    - img_valid flags are AND-ed so a mixed pixel grid is only "valid"
#      if BOTH source images loaded successfully.
#    - CutMix is NEVER applied to validation or test loaders.
# ════════════════════════════════════════════════════════════════════════

def rand_bbox(size, lam):
    """Return (x1, y1, x2, y2) for a CutMix box with area ≈ (1 - lam)."""
    W      = size[-1]
    H      = size[-2]
    cut_w  = int(W * np.sqrt(1.0 - lam))
    cut_h  = int(H * np.sqrt(1.0 - lam))
    cx     = np.random.randint(W)
    cy     = np.random.randint(H)
    x1     = np.clip(cx - cut_w // 2, 0, W)
    y1     = np.clip(cy - cut_h // 2, 0, H)
    x2     = np.clip(cx + cut_w // 2, 0, W)
    y2     = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def cutmix_collate_fn(batch):
    """
    Drop-in collate_fn that applies CutMix with probability CFG.CUTMIX_PROB.
    For non-CutMix batches it behaves identically to the default collate.

    Returned dict keys:
        input_ids, attention_mask, image, img_valid  — standard
        label        — original (y_a) labels
        label_b      — shuffled  (y_b) labels  (same as label if no CutMix)
        lam          — float scalar             (1.0 if no CutMix)
        use_cutmix   — bool flag
    """
    elem = batch[0]
    collated = {
        key: torch.stack([b[key] for b in batch])
        for key in elem
        if isinstance(elem[key], torch.Tensor)
    }

    # Only apply CutMix when labels are present (training batches only)
    if "label" not in collated or np.random.rand() > CFG.CUTMIX_PROB:
        collated["label_b"]    = collated.get("label", None)
        collated["lam"]        = torch.tensor(1.0)
        collated["use_cutmix"] = False
        return collated

    # Sample λ from Beta(1, 1) = Uniform(0, 1)
    lam      = float(np.random.beta(1.0, 1.0))
    B        = collated["image"].size(0)
    rand_idx = torch.randperm(B)

    # Paste patch from shuffled images into current images
    x1, y1, x2, y2 = rand_bbox(collated["image"].shape, lam)
    collated["image"][:, :, y1:y2, x1:x2] = \
        collated["image"][rand_idx, :, y1:y2, x1:x2]

    # Recompute λ from actual pixel-area ratio (matches the paper)
    lam = 1.0 - float((x2 - x1) * (y2 - y1)) / float(
        collated["image"].size(-1) * collated["image"].size(-2)
    )

    # Mixed image is valid only if both source images were valid
    collated["img_valid"] = (
        collated["img_valid"] * collated["img_valid"][rand_idx]
    )

    collated["label_b"]    = collated["label"][rand_idx]
    collated["lam"]        = torch.tensor(lam, dtype=torch.float32)
    collated["use_cutmix"] = True
    return collated


def cutmix_criterion(criterion, logits, batch):
    """
    Compute mixed loss when CutMix was applied, plain loss otherwise.

        loss = lam * CE(logits, y_a) + (1 - lam) * CE(logits, y_b)
    """
    if batch["use_cutmix"]:
        lam = batch["lam"].item()
        return (
            lam       * criterion(logits, batch["label"].to(CFG.DEVICE)) +
            (1 - lam) * criterion(logits, batch["label_b"].to(CFG.DEVICE))
        )
    return criterion(logits, batch["label"].to(CFG.DEVICE))

# ════════════════════════════════════════════════════════════════════════
# DATASET  (full PNG / RGBA support)
# ════════════════════════════════════════════════════════════════════════

class DisasterDataset(Dataset):
    def __init__(self, df, ids, masks, transform, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.ids       = ids
        self.masks     = masks
        self.transform = transform
        self.is_test   = is_test

    def __len__(self): return len(self.df)

    def load_image(self, image_id, transform=None):
        if transform is None:
            transform = self.transform
        path = image_id_to_path(str(image_id))
        try:
            img = read_image(path).float() / 255.0   # (C, H, W), C ∈ {1, 3, 4}

            # ── Grayscale → RGB ──────────────────────────────────────
            if img.shape[0] == 1:
                img = img.repeat(3, 1, 1)

            # ── RGBA → RGB (alpha-composite onto white background) ───
            # Naively dropping the alpha channel creates black fringes
            # wherever the original image was transparent.  Blending
            # onto white is both correct and ImageNet-compatible.
            elif img.shape[0] == 4:
                alpha = img[3:4, :, :]          # (1, H, W) in [0, 1]
                rgb   = img[:3,  :, :]          # (3, H, W)
                white = torch.ones_like(rgb)
                img   = alpha * rgb + (1.0 - alpha) * white  # (3, H, W)

            return transform(img), torch.tensor(1.0)

        except Exception:
            # Return a zero image and mark it invalid so the fusion
            # module ignores the image modality for this sample.
            return (
                torch.zeros(3, CFG.IMG_SIZE, CFG.IMG_SIZE),
                torch.tensor(0.0),
            )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img, valid = self.load_image(row["image_id"])
        item = {
            "input_ids":      torch.from_numpy(self.ids[idx].astype(np.int64)),
            "attention_mask": torch.from_numpy(self.masks[idx].astype(np.int64)),
            "image":          img,
            "img_valid":      valid,
        }
        if not self.is_test:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

# ════════════════════════════════════════════════════════════════════════
# IMAGE ENCODER — Swin-T returning PATCH TOKENS (not pooled vector)
# ════════════════════════════════════════════════════════════════════════

class SwinPatchEncoder(nn.Module):
    """Returns (B, 49, 768) patch-level tokens from Swin-T stage 4."""
    IMG_DIM = 768

    def __init__(self):
        super().__init__()
        base          = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.norm     = base.norm

    def forward(self, x):
        x = self.features(x)   # (B, 7, 7, 768)
        x = self.norm(x)
        B, H, W, C = x.shape
        return x.view(B, H * W, C)   # (B, 49, 768)


# ════════════════════════════════════════════════════════════════════════
# BIDIRECTIONAL CROSS-ATTENTION FUSION
# ════════════════════════════════════════════════════════════════════════

class BidirectionalCrossAttentionFusion(nn.Module):
    def __init__(self, img_dim, txt_dim, fusion_dim, num_heads, dropout):
        super().__init__()
        self.img_proj = nn.Sequential(nn.Linear(img_dim, fusion_dim), nn.LayerNorm(fusion_dim))
        self.txt_proj = nn.Sequential(nn.Linear(txt_dim, fusion_dim), nn.LayerNorm(fusion_dim))
        self.txt2img  = nn.MultiheadAttention(fusion_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_txt2img = nn.LayerNorm(fusion_dim)
        self.img2txt  = nn.MultiheadAttention(fusion_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_img2txt = nn.LayerNorm(fusion_dim)
        self.merge = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, img_patches, txt_tokens, img_valid, txt_key_padding_mask=None):
        img_p = self.img_proj(img_patches)
        txt_p = self.txt_proj(txt_tokens)

        txt_cls   = txt_p[:, 0:1, :]
        attn_a, _ = self.txt2img(query=txt_cls, key=img_p, value=img_p)
        attn_a    = attn_a * img_valid.view(-1, 1, 1)
        out_a     = self.norm_txt2img(txt_cls + attn_a).squeeze(1)

        img_mean  = img_p.mean(dim=1, keepdim=True)
        img_query = torch.where(img_valid.view(-1, 1, 1).bool(), img_mean, txt_cls)
        attn_b, _ = self.img2txt(query=img_query, key=txt_p, value=txt_p,
                                  key_padding_mask=txt_key_padding_mask)
        out_b     = self.norm_img2txt(img_query + attn_b).squeeze(1)

        return self.merge(torch.cat([out_a, out_b], dim=1))


# ════════════════════════════════════════════════════════════════════════
# FULL MODEL
# ════════════════════════════════════════════════════════════════════════

class MultimodalDisasterClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_encoder = SwinPatchEncoder()
        self.txt_encoder = AutoModel.from_pretrained(CFG.TEXT_MODEL)
        txt_hidden       = self.txt_encoder.config.hidden_size
        self.fusion      = BidirectionalCrossAttentionFusion(
            img_dim=SwinPatchEncoder.IMG_DIM, txt_dim=txt_hidden,
            fusion_dim=CFG.FUSION_DIM, num_heads=CFG.ATTN_HEADS, dropout=CFG.DROPOUT,
        )
        self.classifier = nn.Sequential(
            nn.Linear(CFG.FUSION_DIM, CFG.FUSION_DIM // 2),
            nn.LayerNorm(CFG.FUSION_DIM // 2),
            nn.GELU(),
            nn.Dropout(CFG.DROPOUT),
            nn.Linear(CFG.FUSION_DIM // 2, CFG.NUM_CLASSES),
        )

    def forward(self, input_ids, attention_mask, image, img_valid):
        img_patches = self.img_encoder(image)
        txt_out     = self.txt_encoder(input_ids=input_ids, attention_mask=attention_mask)
        txt_tokens  = txt_out.last_hidden_state
        txt_key_padding_mask = (attention_mask == 0)
        fused       = self.fusion(img_patches, txt_tokens, img_valid, txt_key_padding_mask)
        return self.classifier(fused)


# ════════════════════════════════════════════════════════════════════════
# LOSS
# ════════════════════════════════════════════════════════════════════════

class LabelSmoothingLoss(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        nll    = F.nll_loss(log_probs, targets)
        smooth = -log_probs.mean(dim=-1).mean()
        return (1 - CFG.LABEL_SMOOTH) * nll + CFG.LABEL_SMOOTH * smooth


# ════════════════════════════════════════════════════════════════════════
# EMA (Exponential Moving Average)
#
#  Usage:
#    ema = EMA(model, decay=CFG.EMA_DECAY)
#    after each optimizer step: ema.update(model)
#    for validation/inference:  ema.apply_shadow(model)
#    to restore training weights: ema.restore(model)
# ════════════════════════════════════════════════════════════════════════

class EMA:
    """
    Maintains a shadow copy of model parameters as an exponential
    moving average.  Shadow weights are used for evaluation; the live
    weights continue to train normally.

    decay = 0.999 means each update keeps 99.9 % of the old shadow
    and blends in 0.1 % of the new live weights, smoothing batch noise
    while still tracking genuine improvements.
    """
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay  = decay
        self.shadow = {
            name: param.data.clone()
            for name, param in model.named_parameters()
            if param.requires_grad
        }

    @torch.no_grad()
    def update(self, model: nn.Module):
        """Call after every optimizer.step() to blend live → shadow."""
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(
                    param.data, alpha=1.0 - self.decay
                )

    def apply_shadow(self, model: nn.Module):
        """Swap live weights out and shadow weights in (for eval/test)."""
        self._backup = {
            name: param.data.clone()
            for name, param in model.named_parameters()
            if param.requires_grad
        }
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name])

    def restore(self, model: nn.Module):
        """Restore the live weights after eval/test."""
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self._backup[name])
        del self._backup


# ════════════════════════════════════════════════════════════════════════
# LLRD OPTIMIZER
#
#  Layer-wise Learning Rate Decay for both BanglaBERT and Swin-T.
#
#  BanglaBERT:
#    embeddings         → TXT_LR * alpha^(num_layers)   (lowest)
#    encoder.layer[0]   → TXT_LR * alpha^(num_layers-1)
#    ...
#    encoder.layer[N-1] → TXT_LR * alpha^1
#    pooler / other     → TXT_LR                        (highest)
#
#  Swin-T (features[0..7], two sub-blocks per stage):
#    features[0-1]      → IMG_LR * alpha^3   stage 1 (lowest)
#    features[2-3]      → IMG_LR * alpha^2   stage 2
#    features[4-5]      → IMG_LR * alpha^1   stage 3
#    features[6-7]      → IMG_LR * alpha^0   stage 4 (highest)
#    norm               → IMG_LR
#
#  Head (fusion + classifier) → HEAD_LR always.
# ════════════════════════════════════════════════════════════════════════

def build_llrd_optimizer(model: nn.Module) -> AdamW:
    no_decay = {"bias", "LayerNorm.weight"}
    param_groups = []

    def add_group(named_params, lr):
        wd_params    = [p for n, p in named_params
                        if not any(nd in n for nd in no_decay) and p.requires_grad]
        no_wd_params = [p for n, p in named_params
                        if     any(nd in n for nd in no_decay) and p.requires_grad]
        if wd_params:
            param_groups.append({"params": wd_params,    "lr": lr, "weight_decay": CFG.WEIGHT_DECAY})
        if no_wd_params:
            param_groups.append({"params": no_wd_params, "lr": lr, "weight_decay": 0.0})

    # ── BanglaBERT LLRD ─────────────────────────────────────────────
    txt_encoder    = model.txt_encoder
    num_txt_layers = txt_encoder.config.num_hidden_layers  # typically 12

    emb_lr = CFG.TXT_LR * (CFG.LLRD_ALPHA ** num_txt_layers)
    add_group(list(txt_encoder.embeddings.named_parameters()), emb_lr)

    for layer_idx, layer in enumerate(txt_encoder.encoder.layer):
        layer_lr = CFG.TXT_LR * (CFG.LLRD_ALPHA ** (num_txt_layers - 1 - layer_idx))
        add_group(list(layer.named_parameters()), layer_lr)

    pooler_params = [(n, p) for n, p in txt_encoder.named_parameters()
                     if "embeddings" not in n and "encoder.layer" not in n]
    add_group(pooler_params, CFG.TXT_LR)

    # ── Swin-T LLRD ─────────────────────────────────────────────────
    swin_encoder    = model.img_encoder
    num_swin_stages = 4

    for stage_idx in range(num_swin_stages):
        sub_blocks = [swin_encoder.features[stage_idx * 2],
                      swin_encoder.features[stage_idx * 2 + 1]]
        stage_lr   = CFG.IMG_LR * (CFG.LLRD_ALPHA ** (num_swin_stages - 1 - stage_idx))
        named_params = []
        for blk in sub_blocks:
            named_params.extend(list(blk.named_parameters()))
        add_group(named_params, stage_lr)

    add_group(list(swin_encoder.norm.named_parameters()), CFG.IMG_LR)

    # ── Head ────────────────────────────────────────────────────────
    head_params = (list(model.fusion.named_parameters()) +
                   list(model.classifier.named_parameters()))
    add_group(head_params, CFG.HEAD_LR)

    return AdamW(param_groups)


# ════════════════════════════════════════════════════════════════════════
# TRAIN EPOCH  (CutMix-aware)
# ════════════════════════════════════════════════════════════════════════

def train_epoch(model, ema, loader, optimizer, scheduler, scaler, criterion):
    model.train()
    total_loss = 0
    preds_all, labels_all = [], []

    for batch in tqdm(loader, desc="Training"):
        ids    = batch["input_ids"].to(CFG.DEVICE, non_blocking=True)
        mask   = batch["attention_mask"].to(CFG.DEVICE, non_blocking=True)
        img    = batch["image"].to(CFG.DEVICE, non_blocking=True).contiguous(
                     memory_format=torch.channels_last)
        valid  = batch["img_valid"].to(CFG.DEVICE, non_blocking=True)
        labels = batch["label"].to(CFG.DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with autocast(device_type="cuda", dtype=torch.float16, enabled=CFG.AMP):
            logits = model(ids, mask, img, valid)
            # Use CutMix-aware loss (falls back to plain CE when not applied)
            loss   = cutmix_criterion(criterion, logits, batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Update EMA shadow weights after each optimizer step
        ema.update(model)

        total_loss += loss.item()
        preds_all.extend(logits.argmax(1).detach().cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    return total_loss / len(loader), accuracy_score(labels_all, preds_all)


# ════════════════════════════════════════════════════════════════════════
# EVAL EPOCH  (uses EMA shadow weights)
# ════════════════════════════════════════════════════════════════════════

def eval_epoch(model, ema, loader, criterion):
    """
    Swap in EMA shadow weights, run evaluation, then restore live weights.
    Every validation score reflects the smoothed model.
    """
    ema.apply_shadow(model)
    model.eval()
    total_loss = 0
    preds_all, labels_all, probs_all = [], [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            ids    = batch["input_ids"].to(CFG.DEVICE, non_blocking=True)
            mask   = batch["attention_mask"].to(CFG.DEVICE, non_blocking=True)
            img    = batch["image"].to(CFG.DEVICE, non_blocking=True).contiguous(
                         memory_format=torch.channels_last)
            valid  = batch["img_valid"].to(CFG.DEVICE, non_blocking=True)
            labels = batch["label"].to(CFG.DEVICE, non_blocking=True)

            with autocast(device_type="cuda", dtype=torch.float16, enabled=CFG.AMP):
                logits = model(ids, mask, img, valid)
                loss   = criterion(logits, labels)

            total_loss += loss.item()
            probs = torch.softmax(logits, dim=1)
            probs_all.extend(probs.cpu().numpy())
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    ema.restore(model)
    return total_loss / len(loader), accuracy_score(labels_all, preds_all), np.array(probs_all)


# ════════════════════════════════════════════════════════════════════════
# TTA INFERENCE
#
#  For each of the 5 TTA views:
#    1. Temporarily override dataset.transform with the view's transform.
#    2. Forward pass through the EMA model.
#    3. Accumulate softmax probabilities.
#  Average probabilities across all views, then argmax.
# ════════════════════════════════════════════════════════════════════════

def tta_inference(model, ema, dataset, batch_size):
    """
    Run TTA inference over `dataset`.
    Returns np.ndarray of shape (N, NUM_CLASSES) — averaged softmax probs.
    """
    ema.apply_shadow(model)
    model.eval()

    n           = len(dataset)
    accumulated = np.zeros((n, CFG.NUM_CLASSES), dtype=np.float32)

    for view_idx, tfm in enumerate(tta_transforms):
        original_transform = dataset.transform
        dataset.transform  = tfm

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True,
        )

        view_probs = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=f"TTA view {view_idx + 1}/{len(tta_transforms)}"):
                ids   = batch["input_ids"].to(CFG.DEVICE, non_blocking=True)
                mask  = batch["attention_mask"].to(CFG.DEVICE, non_blocking=True)
                img   = batch["image"].to(CFG.DEVICE, non_blocking=True).contiguous(
                            memory_format=torch.channels_last)
                valid = batch["img_valid"].to(CFG.DEVICE, non_blocking=True)

                with autocast(device_type="cuda", dtype=torch.float16, enabled=CFG.AMP):
                    logits = model(ids, mask, img, valid)

                view_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())

        accumulated      += np.array(view_probs)
        dataset.transform = original_transform

    ema.restore(model)
    return accumulated / len(tta_transforms)


# ════════════════════════════════════════════════════════════════════════
# K-FOLD TRAINING  (5-Fold Stratified CV)
# ════════════════════════════════════════════════════════════════════════

skf        = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds  = np.zeros((len(train_df), CFG.NUM_CLASSES))
test_probs = np.zeros((len(test_df),  CFG.NUM_CLASSES))
criterion  = LabelSmoothingLoss()

for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df["clean_text"], train_df["label"])):
    print(f"\n{'='*20} FOLD {fold + 1} / {CFG.N_FOLDS} {'='*20}")

    tr_df, vl_df = train_df.iloc[tr_idx], train_df.iloc[vl_idx]

    tr_ds = DisasterDataset(tr_df,   TRAIN_IDS[tr_idx], TRAIN_MASK[tr_idx], train_transforms)
    vl_ds = DisasterDataset(vl_df,   TRAIN_IDS[vl_idx], TRAIN_MASK[vl_idx], val_transforms)
    te_ds = DisasterDataset(test_df, TEST_IDS,           TEST_MASK,          val_transforms, is_test=True)

    # Training loader uses CutMix collate_fn; val/test use default collate
    tr_loader = DataLoader(
        tr_ds,
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True,
        collate_fn=cutmix_collate_fn,      # ← CutMix applied here only
    )
    vl_loader = DataLoader(
        vl_ds,
        batch_size=CFG.BATCH_SIZE * 2,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True,
    )

    model = MultimodalDisasterClassifier().to(CFG.DEVICE)
    model = model.to(memory_format=torch.channels_last)
    try:
        model = torch.compile(model)
    except Exception:
        pass

    optimizer    = build_llrd_optimizer(model)
    total_steps  = len(tr_loader) * CFG.EPOCHS
    warmup_steps = int(total_steps * CFG.WARMUP_RATIO)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler(device="cuda")
    ema          = EMA(model, decay=CFG.EMA_DECAY)

    best_acc = 0
    patience = 0

    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch + 1}")

        tr_loss, tr_acc              = train_epoch(model, ema, tr_loader, optimizer,
                                                   scheduler, scaler, criterion)
        vl_loss, vl_acc, vl_probs   = eval_epoch(model, ema, vl_loader, criterion)

        print(f"Train  Loss: {tr_loss:.4f} | Train  Acc: {tr_acc:.4f}")
        print(f"Val    Loss: {vl_loss:.4f} | Val    Acc: {vl_acc:.4f}  [EMA]")

        if vl_acc > best_acc:
            best_acc = vl_acc
            torch.save({
                "model_state": model.state_dict(),
                "ema_shadow":  ema.shadow,
            }, f"{CFG.OUTPUT_DIR}/best_fold_{fold}.pt")
            patience = 0
        else:
            patience += 1
            if patience >= CFG.PATIENCE:
                print("Early stopping triggered.")
                break

    # Load best checkpoint and restore EMA shadow
    checkpoint = torch.load(f"{CFG.OUTPUT_DIR}/best_fold_{fold}.pt")
    model.load_state_dict(checkpoint["model_state"])
    ema.shadow = checkpoint["ema_shadow"]

    # OOF probabilities with TTA + EMA
    print("\nCollecting OOF probs with TTA…")
    oof_preds[vl_idx] = tta_inference(model, ema, vl_ds, CFG.BATCH_SIZE * 2)

    # Test probabilities with TTA + EMA
    print("Collecting test probs with TTA…")
    fold_test_probs = tta_inference(model, ema, te_ds, CFG.BATCH_SIZE * 2)
    test_probs     += fold_test_probs / CFG.N_FOLDS

    del model, ema
    gc.collect()
    torch.cuda.empty_cache()

# ════════════════════════════════════════════════════════════════════════
# FINAL OOF
# ════════════════════════════════════════════════════════════════════════

oof_acc = accuracy_score(train_df["label"].values, np.argmax(oof_preds, axis=1))
print(f"\n{'='*38}")
print(f"  FINAL OOF Accuracy: {oof_acc:.4f}  (5-Fold + EMA + TTA + CutMix)")
print(f"{'='*38}")

# ════════════════════════════════════════════════════════════════════════
# SUBMISSION
# ════════════════════════════════════════════════════════════════════════

preds      = np.argmax(test_probs, axis=1)
classes    = le.inverse_transform(preds)
submission = pd.DataFrame({"image_id": test_df["image_id"], "category": classes})
submission.to_csv(f"{CFG.OUTPUT_DIR}/submission.csv", index=False)
print("\nsubmission.csv saved!")
print(submission.head())